## Cell 1 — ติดตั้งและ Import

In [1222]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib
import ta
from src.data_loader import get_processed_data

# ดึง M1 ย้อนหลัง 7 วัน (~10,000 แท่ง)
df = get_processed_data('XAUUSD', interval='1m', num_bars=10000)
print(f'ข้อมูลทั้งหมด: {len(df)} แถว')
df.tail(3)

[Data] กำลังดึง XAUUSD จาก MT5 | interval=1m | แท่งย้อนหลัง=10000 ...
[Data] ดึงข้อมูลสำเร็จ → 10000 แท่งเทียน
[Feature] กำลังสร้าง Indicators สำหรับ Scalping ...
[Feature] ลบ NaN ออก 33 แถว → เหลือ 9967 แถว
ข้อมูลทั้งหมด: 9967 แถว


,open,high,low,close,volume,spread,real_volume,returns,log_returns,rsi_7,macd_diff,atr_5,bb_width,bb_position,dist_to_ema9,ema_cross,log_returns_lag_1,log_returns_lag_2,log_returns_lag_3,target
time,,,,,,,,,,,,,,,,,,,,
2026-05-26 22:37:00,4504.13,4504.19,4503.45,4503.80,174,4,0,-0.000049,-0.000049,58.463004,-0.019051,1.015894,0.000823,0.733585,0.000102,0.670164,-0.000042,0.000142,0.000222,0
2026-05-26 22:38:00,4503.85,4503.85,4503.52,4503.68,167,6,0,-0.000027,-0.000027,56.762457,-0.031118,0.878715,0.000715,0.662715,0.000060,0.646261,-0.000049,-0.000042,0.000142,0
2026-05-26 22:39:00,4503.66,4503.94,4503.13,4503.36,162,4,0,-0.000071,-0.000071,52.052026,-0.064970,0.864972,0.000648,0.527765,-0.000009,0.582218,-0.000027,-0.000049,-0.000042,0


## Cell 2 — Feature Engineering สำหรับ Scalping M1

In [1223]:
# RSI สั้นมาก
df['rsi_3']  = ta.momentum.RSIIndicator(df['close'], window=3).rsi()
df['rsi_7']  = ta.momentum.RSIIndicator(df['close'], window=7).rsi()

# EMA สั้น สำหรับ trend M1
df['ema3']  = df['close'].ewm(span=3).mean()
df['ema7']  = df['close'].ewm(span=7).mean()
df['ema21'] = df['close'].ewm(span=21).mean()
df['ema_fast_cross'] = (df['ema3'] - df['ema7']) / df['close']
df['ema_trend']      = (df['ema7'] - df['ema21']) / df['close']

# ATR สั้น (volatility)
df['atr_3']   = ta.volatility.AverageTrueRange(df['high'], df['low'], df['close'], window=3).average_true_range()
df['atr_7']   = ta.volatility.AverageTrueRange(df['high'], df['low'], df['close'], window=7).average_true_range()
df['atr_ratio'] = df['atr_3'] / df['atr_7']  # > 1 = ตลาดร้อน

# Price Action
df['candle_body'] = abs(df['close'] - df['open']) / df['atr_7']
df['upper_wick']  = (df['high'] - df[['close','open']].max(axis=1)) / df['atr_7']
df['lower_wick']  = (df[['close','open']].min(axis=1) - df['low']) / df['atr_7']
df['is_bullish']  = (df['close'] > df['open']).astype(int)

# Momentum lag 1-5 แท่ง
for i in range(1, 6):
    df[f'ret_lag_{i}'] = df['close'].pct_change(i)

# Volume spike
if 'volume' in df.columns and df['volume'].sum() > 0:
    df['vol_ratio'] = df['volume'] / df['volume'].rolling(20).mean()
else:
    df['vol_ratio'] = 1.0

df.dropna(inplace=True)
print(f'หลัง Feature Engineering: {len(df)} แถว | {len(df.columns)} columns')

หลัง Feature Engineering: 9948 แถว | 39 columns


## Cell 3 — สร้าง Target (ปรับได้)

In [ ]:
# ปรับ 2 ค่านี้ได้ตามต้องการ
FORWARD_BARS = 100      # มองข้างหน้า 3 แท่ง M1
TARGET_POINTS = 3   # ราคาต้องเคลื่อน >= 5 points ($0.50)

future_close = df['close'].shift(-FORWARD_BARS)
df['target'] = np.where(
    future_close >= df['close'] + TARGET_POINTS * 0.01, 1,    # BUY signal
    np.where(
        future_close <= df['close'] - TARGET_POINTS * 0.01, 0, # SELL signal
        np.nan  # NEUTRAL — ตัดทิ้ง ไม่เทรด
    )
)

df.dropna(subset=['target'], inplace=True)
df['target'] = df['target'].astype(int)

print(f'ข้อมูลที่ใช้ Train: {len(df)} แถว')
print(f'BUY (1):  {(df.target==1).sum()} ({(df.target==1).mean()*100:.1f}%)')
print(f'SELL (0): {(df.target==0).sum()} ({(df.target==0).mean()*100:.1f}%)')

ข้อมูลที่ใช้ Train: 9819 แถว
BUY (1):  4563 (46.5%)
SELL (0): 5256 (53.5%)


## Cell 4 — เลือก Features & Split

In [1225]:
FEATURES = [
    'rsi_3', 'rsi_7',
    'ema_fast_cross', 'ema_trend',
    'atr_3', 'atr_7', 'atr_ratio',
    'candle_body', 'upper_wick', 'lower_wick', 'is_bullish',
    'ret_lag_1', 'ret_lag_2', 'ret_lag_3', 'ret_lag_4', 'ret_lag_5',
    'vol_ratio',
    'macd_diff', 'bb_width', 'bb_position',
]

# เอาเฉพาะที่มีในข้อมูลจริง
FEATURES = [f for f in FEATURES if f in df.columns]
print(f'Features ที่ใช้: {len(FEATURES)} ตัว')
print(FEATURES)

X = df[FEATURES]
y = df['target']

# Time-series split — ห้าม shuffle!
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f'\nTrain: {len(X_train)} แถว | Test: {len(X_test)} แถว')

Features ที่ใช้: 20 ตัว
['rsi_3', 'rsi_7', 'ema_fast_cross', 'ema_trend', 'atr_3', 'atr_7', 'atr_ratio', 'candle_body', 'upper_wick', 'lower_wick', 'is_bullish', 'ret_lag_1', 'ret_lag_2', 'ret_lag_3', 'ret_lag_4', 'ret_lag_5', 'vol_ratio', 'macd_diff', 'bb_width', 'bb_position']

Train: 7855 แถว | Test: 1964 แถว


## Cell 5 — Train XGBoost (โหดขึ้น)

In [1226]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

# Balanced weight กัน imbalance
sample_weights = compute_sample_weight('balanced', y_train)

model = XGBClassifier(
    n_estimators=1000,         # เพิ่มจาก 300 → 1000 (ใช้ early stopping)
    max_depth=5,               # ลึกขึ้น (4→5)
    learning_rate=0.02,        # ช้าลง = แม่นขึ้น
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=5,        # กัน overfit บน noise M1
    gamma=0.1,                 # ต้องการ gain ขั้นต่ำก่อน split
    reg_alpha=0.1,             # L1 regularization
    reg_lambda=1.5,            # L2 regularization
    eval_metric='logloss',
    random_state=42,
    early_stopping_rounds=50,  # หยุดถ้า 50 รอบไม่ดีขึ้น
)

model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    verbose=100
)

print(f'\n✅ Train เสร็จ! Best iteration: {model.best_iteration}')

[0]	validation_0-logloss:0.69262
[79]	validation_0-logloss:0.68891

✅ Train เสร็จ! Best iteration: 29


## Cell 6 — วัดผล (Precision สำคัญกว่า Accuracy!)

In [1227]:
from sklearn.metrics import classification_report, precision_score

y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['SELL (0)', 'BUY (1)']))

buy_precision  = precision_score(y_test, y_pred, pos_label=1)
sell_precision = precision_score(y_test, y_pred, pos_label=0)
print(f'🎯 BUY Precision:  {buy_precision*100:.1f}%')
print(f'🎯 SELL Precision: {sell_precision*100:.1f}%')

# สำหรับ Scalping: Precision > 55% = มี Edge
if buy_precision > 0.55 and sell_precision > 0.55:
    print('\n✅ โมเดลผ่านเกณฑ์ Scalping (>55%)')
else:
    print('\n⚠️ ยังไม่ผ่าน — ลองเพิ่ม TARGET_POINTS หรือ FORWARD_BARS ใน Cell 3')

              precision    recall  f1-score   support

    SELL (0)       0.63      0.79      0.70      1143
     BUY (1)       0.55      0.36      0.44       821

    accuracy                           0.61      1964
   macro avg       0.59      0.58      0.57      1964
weighted avg       0.60      0.61      0.59      1964



🎯 BUY Precision:  55.2%
🎯 SELL Precision: 63.3%

✅ โมเดลผ่านเกณฑ์ Scalping (>55%)


## Cell 7 — Confidence Filter (เทรดเฉพาะเมื่อมั่นใจ)

In [1228]:
CONFIDENCE_THRESHOLD = 0.60

high_conf = (y_pred_prob >= CONFIDENCE_THRESHOLD) | (y_pred_prob <= 1 - CONFIDENCE_THRESHOLD)

print(f'สัญญาณที่ผ่าน Confidence > {CONFIDENCE_THRESHOLD*100:.0f}%: {high_conf.sum()}/{len(y_test)}')

# ✅ เช็คก่อนว่ามีข้อมูลพอ
if high_conf.sum() == 0:
    print('⚠️ ไม่มีสัญญาณผ่าน threshold นี้เลย — ลด threshold ลงครับ')
else:
    y_hc_true = y_test[high_conf]
    y_hc_pred = y_pred[high_conf]
    
    hc_buy_prec  = precision_score(y_hc_true, y_hc_pred, pos_label=1)
    hc_sell_prec = precision_score(y_hc_true, y_hc_pred, pos_label=0)
    print(f'  BUY Precision:  {hc_buy_prec*100:.1f}%')
    print(f'  SELL Precision: {hc_sell_prec*100:.1f}%')

สัญญาณที่ผ่าน Confidence > 60%: 208/1964
  BUY Precision:  0.0%
  SELL Precision: 60.1%


## Cell 8 — บันทึกโมเดล + Config

In [1229]:
import os

os.makedirs('models', exist_ok=True)

# บันทึกโมเดล
joblib.dump(model, 'models/tuned_trading_model.joblib')

# บันทึก config (main.py จะอ่านค่านี้)
config = {
    'features': FEATURES,
    'confidence_threshold': CONFIDENCE_THRESHOLD,
    'forward_bars': FORWARD_BARS,
    'target_points': TARGET_POINTS,
    'interval': '1m',
}
joblib.dump(config, 'models/model_config.joblib')

print('✅ บันทึกเสร็จแล้ว!')
print(f'   Features: {len(FEATURES)} ตัว')
print(f'   Confidence threshold: {CONFIDENCE_THRESHOLD}')
print(f'\n⚠️  Re-Train ใหม่ทุก 2-3 วัน เพราะ M1 เปลี่ยนเร็วมาก!')

✅ บันทึกเสร็จแล้ว!
   Features: 20 ตัว
   Confidence threshold: 0.6

⚠️  Re-Train ใหม่ทุก 2-3 วัน เพราะ M1 เปลี่ยนเร็วมาก!
